# OJP Pilot und finale Stationskonfiguration

Dieses Notebook enthält nur noch den bereinigten, wiederverwendbaren Code für die OJP-Datensammlung und die final ausgewählten Beobachtungspunkte.

**Ausführungsreihenfolge:**
1. Imports, API-Token und URL laden
2. `collect_ojp_station()` definieren
3. `find_ojp_stations()` definieren
4. Finale Rail- und Local-Stationslisten laden
5. Gemeinsame Stationskonfiguration prüfen

Die bisherigen Such-, Debug- und Einzeltest-Zellen wurden entfernt. Der API-Token wird nur über `getpass()` eingegeben und nicht im Notebook gespeichert.

## 1. Imports und API-Konfiguration

In [5]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from uuid import uuid4
from xml.sax.saxutils import escape

URL = "https://api.opentransportdata.swiss/ojp20"

TOKEN = getpass("OJP API Token: ").strip()

if not TOKEN:
    raise ValueError("Der OJP API Token ist leer.")


## 2. OJP-Daten für eine Haltestelle sammeln

In [6]:
def collect_ojp_station(
    stop_id,
    stop_name,
    token,
    number_of_results=30
):
    """
    Ruft OJP-Echtzeitdaten für eine Haltestelle ab,
    parst die XML-Antwort und gibt einen pandas DataFrame zurück.
    """

    # 1. Zeitpunkt des API-Abrufs
    now_utc = datetime.now(timezone.utc)
    timestamp = (
        now_utc
        .isoformat(timespec="milliseconds")
        .replace("+00:00", "Z")
    )
    message_id = f"zhaw-project-{uuid4()}"
    stop_name_xml = escape(stop_name)

    # 2. XML Request
    xml_request = f"""<?xml version="1.0" encoding="UTF-8"?>
<OJP
    xmlns="http://www.vdv.de/ojp"
    xmlns:siri="http://www.siri.org.uk/siri"
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    xmlns:xsd="http://www.w3.org/2001/XMLSchema"
    xsi:schemaLocation="http://www.vdv.de/ojp"
    version="2.0">

    <OJPRequest>
        <siri:ServiceRequest>

            <siri:ServiceRequestContext>
                <siri:Language>de</siri:Language>
            </siri:ServiceRequestContext>

            <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>
            <siri:RequestorRef>ZHAW_DataAnalytics_Project</siri:RequestorRef>

            <OJPStopEventRequest>

                <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>
                <siri:MessageIdentifier>{message_id}</siri:MessageIdentifier>

                <Location>
                    <PlaceRef>
                        <siri:StopPointRef>{stop_id}</siri:StopPointRef>
                        <Name>
                            <Text>{stop_name_xml}</Text>
                        </Name>
                    </PlaceRef>
                    <DepArrTime>{timestamp}</DepArrTime>
                </Location>

                <Params>
                    <NumberOfResults>{number_of_results}</NumberOfResults>
                    <StopEventType>departure</StopEventType>
                    <IncludePreviousCalls>false</IncludePreviousCalls>
                    <IncludeOnwardCalls>false</IncludeOnwardCalls>
                    <UseRealtimeData>full</UseRealtimeData>
                </Params>

            </OJPStopEventRequest>

        </siri:ServiceRequest>
    </OJPRequest>

</OJP>
"""

    # 3. API Request
    headers = {
        "Content-Type": "application/xml",
        "Authorization": f"Bearer {token}"
    }

    response = requests.post(
        URL,
        headers=headers,
        data=xml_request.encode("utf-8"),
        timeout=30
    )

    print(f"{stop_name}: HTTP {response.status_code}")

    if response.status_code != 200:
        print("\nAPI-Fehler:")
        print(response.text[:1000])

    response.raise_for_status()

    # 4. Raw XML speichern
    safe_station_name = (
        stop_name
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace(",", "")
        .replace("ü", "ue")
        .replace("ö", "oe")
        .replace("ä", "ae")
        .replace("é", "e")
        .replace("è", "e")
    )

    raw_dir = Path("data/raw/ojp")
    raw_dir.mkdir(parents=True, exist_ok=True)

    raw_filename = (
        raw_dir
        / f"{safe_station_name}_{now_utc.strftime('%Y%m%d_%H%M%S')}.xml"
    )
    raw_filename.write_bytes(response.content)

    # 5. XML parsen
    root = ET.fromstring(response.content)

    ns = {
        "ojp": "http://www.vdv.de/ojp",
        "siri": "http://www.siri.org.uk/siri"
    }

    results = root.findall(".//ojp:StopEventResult", ns)
    print(f"{stop_name}: {len(results)} Verbindungen")

    # 6. Daten extrahieren
    rows = []

    for result in results:
        stop_event = result.find("ojp:StopEvent", ns)

        if stop_event is None:
            continue

        this_call = stop_event.find(
            "ojp:ThisCall/ojp:CallAtStop",
            ns
        )
        service = stop_event.find(
            "ojp:Service",
            ns
        )

        if this_call is None or service is None:
            continue

        rows.append({
            "collection_timestamp": timestamp,
            "station_id": stop_id,
            "station_name": stop_name,
            "stop_point_ref": this_call.findtext(
                "siri:StopPointRef",
                default=None,
                namespaces=ns
            ),
            "operating_day": service.findtext(
                "ojp:OperatingDayRef",
                default=None,
                namespaces=ns
            ),
            "journey_ref": service.findtext(
                "ojp:JourneyRef",
                default=None,
                namespaces=ns
            ),
            "transport_mode": service.findtext(
                "ojp:Mode/ojp:PtMode",
                default=None,
                namespaces=ns
            ),
            "product_category": service.findtext(
                "ojp:ProductCategory/ojp:Name/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "public_code": service.findtext(
                "ojp:PublicCode",
                default=None,
                namespaces=ns
            ),
            "line": service.findtext(
                "ojp:PublishedServiceName/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "train_number": service.findtext(
                "ojp:TrainNumber",
                default=None,
                namespaces=ns
            ),
            "origin": service.findtext(
                "ojp:OriginText/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "destination": service.findtext(
                "ojp:DestinationText/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "planned_platform": this_call.findtext(
                "ojp:PlannedQuay/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "estimated_platform": this_call.findtext(
                "ojp:EstimatedQuay/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "scheduled_departure": this_call.findtext(
                "ojp:ServiceDeparture/ojp:TimetabledTime",
                default=None,
                namespaces=ns
            ),
            "estimated_departure": this_call.findtext(
                "ojp:ServiceDeparture/ojp:EstimatedTime",
                default=None,
                namespaces=ns
            )
        })

    # 7. DataFrame erstellen
    df = pd.DataFrame(rows)

    if df.empty:
        print(f"Keine Daten für {stop_name} gefunden.")
        return df

    # 8. Zeitvariablen
    time_columns = [
        "collection_timestamp",
        "scheduled_departure",
        "estimated_departure"
    ]

    for column in time_columns:
        df[column] = pd.to_datetime(
            df[column],
            utc=True,
            errors="coerce"
        )

    # 9. Realtime und Delay
    df["has_realtime"] = df["estimated_departure"].notna()

    # Fehlende EstimatedTime bleibt NaN und wird nicht als 0 Minuten interpretiert.
    df["predicted_delay_minutes"] = (
        df["estimated_departure"]
        - df["scheduled_departure"]
    ).dt.total_seconds() / 60

    # 10. Schweizer Lokalzeit
    df["collection_timestamp_local"] = (
        df["collection_timestamp"].dt.tz_convert("Europe/Zurich")
    )
    df["scheduled_departure_local"] = (
        df["scheduled_departure"].dt.tz_convert("Europe/Zurich")
    )
    df["estimated_departure_local"] = (
        df["estimated_departure"].dt.tz_convert("Europe/Zurich")
    )

    # 11. Zeitliche Merkmale
    df["date"] = df["scheduled_departure_local"].dt.date
    df["hour"] = df["scheduled_departure_local"].dt.hour
    df["weekday"] = df["scheduled_departure_local"].dt.day_name()
    df["weekend"] = df["scheduled_departure_local"].dt.dayofweek >= 5
    df["minutes_until_departure"] = (
        df["scheduled_departure"]
        - df["collection_timestamp"]
    ).dt.total_seconds() / 60

    # 12. CSV Snapshot speichern
    interim_dir = Path("data/interim")
    interim_dir.mkdir(parents=True, exist_ok=True)

    csv_filename = (
        interim_dir
        / f"{safe_station_name}_{now_utc.strftime('%Y%m%d_%H%M%S')}.csv"
    )

    df.to_csv(csv_filename, index=False)
    print(f"Gespeichert: {csv_filename}")

    return df


## 3. Haltestellen über Namen suchen

In [7]:
def find_ojp_stations(
    search_name,
    token,
    number_of_results=5
):
    """
    Sucht OJP-Haltestellen anhand eines Namens
    und gibt die gefundenen Stop-IDs als DataFrame zurück.
    """

    # 1. Zeitpunkt und Message-ID
    now_utc = datetime.now(timezone.utc)
    timestamp = (
        now_utc
        .isoformat(timespec="milliseconds")
        .replace("+00:00", "Z")
    )
    message_id = f"zhaw-location-{uuid4()}"
    search_name_xml = escape(search_name)

    # 2. OJP LocationInformationRequest
    xml_request = f"""<?xml version="1.0" encoding="UTF-8"?>
<OJP
    xmlns="http://www.vdv.de/ojp"
    xmlns:siri="http://www.siri.org.uk/siri"
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    version="2.0">

    <OJPRequest>
        <siri:ServiceRequest>

            <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>
            <siri:RequestorRef>ZHAW_DataAnalytics_Project</siri:RequestorRef>

            <OJPLocationInformationRequest>

                <siri:RequestTimestamp>{timestamp}</siri:RequestTimestamp>
                <siri:MessageIdentifier>{message_id}</siri:MessageIdentifier>

                <InitialInput>
                    <Name>{search_name_xml}</Name>
                </InitialInput>

                <Restrictions>
                    <Type>stop</Type>
                    <NumberOfResults>{number_of_results}</NumberOfResults>
                    <IncludePtModes>true</IncludePtModes>
                </Restrictions>

            </OJPLocationInformationRequest>

        </siri:ServiceRequest>
    </OJPRequest>

</OJP>
"""

    # 3. API Request
    headers = {
        "Content-Type": "application/xml",
        "Authorization": f"Bearer {token}"
    }

    response = requests.post(
        URL,
        headers=headers,
        data=xml_request.encode("utf-8"),
        timeout=30
    )

    print(
        f"Stationssuche '{search_name}': "
        f"HTTP {response.status_code}"
    )

    if response.status_code != 200:
        print("\nAPI-Fehler:")
        print(response.text[:1000])

    response.raise_for_status()

    # 4. XML parsen
    root = ET.fromstring(response.content)

    ns = {
        "ojp": "http://www.vdv.de/ojp",
        "siri": "http://www.siri.org.uk/siri"
    }

    place_results = root.findall(
        ".//ojp:PlaceResult",
        ns
    )

    # 5. Treffer extrahieren
    rows = []

    for result in place_results:
        place = result.find("ojp:Place", ns)

        if place is None:
            continue

        stop_place = place.find("ojp:StopPlace", ns)

        if stop_place is None:
            continue

        modes = []

        for mode in place.findall("ojp:Mode", ns):
            pt_mode = mode.findtext(
                "ojp:PtMode",
                default=None,
                namespaces=ns
            )

            if pt_mode is not None:
                modes.append(pt_mode)

        rows.append({
            "stop_id": stop_place.findtext(
                "ojp:StopPlaceRef",
                default=None,
                namespaces=ns
            ),
            "stop_name": stop_place.findtext(
                "ojp:StopPlaceName/ojp:Text",
                default=None,
                namespaces=ns
            ),
            "modes": ", ".join(modes),
            "latitude": place.findtext(
                "ojp:GeoPosition/siri:Latitude",
                default=None,
                namespaces=ns
            ),
            "longitude": place.findtext(
                "ojp:GeoPosition/siri:Longitude",
                default=None,
                namespaces=ns
            ),
            "probability": result.findtext(
                "ojp:Probability",
                default=None,
                namespaces=ns
            ),
            "complete": result.findtext(
                "ojp:Complete",
                default=None,
                namespaces=ns
            )
        })

    # 6. DataFrame
    df_stations = pd.DataFrame(rows)

    if not df_stations.empty:
        df_stations["latitude"] = pd.to_numeric(
            df_stations["latitude"],
            errors="coerce"
        )
        df_stations["longitude"] = pd.to_numeric(
            df_stations["longitude"],
            errors="coerce"
        )
        df_stations["probability"] = pd.to_numeric(
            df_stations["probability"],
            errors="coerce"
        )

    print("Gefundene Haltestellen:", len(df_stations))

    return df_stations


## 4. Finale Stationsauswahl

In [8]:
rail_stations = [
    {"city": "Zürich",     "stop_id": "ch:1:sloid:3000",  "stop_name": "Zürich HB"},
    {"city": "Bern",       "stop_id": "ch:1:sloid:7000",  "stop_name": "Bern"},
    {"city": "Basel",      "stop_id": "ch:1:sloid:10",    "stop_name": "Basel SBB"},
    {"city": "Luzern",     "stop_id": "ch:1:sloid:5000",  "stop_name": "Luzern"},
    {"city": "St. Gallen", "stop_id": "ch:1:sloid:6302",  "stop_name": "St. Gallen"},
    {"city": "Lausanne",   "stop_id": "ch:1:sloid:1120",  "stop_name": "Lausanne"},
    {"city": "Genève",     "stop_id": "ch:1:sloid:1008",  "stop_name": "Genève"},
    {"city": "Lugano",     "stop_id": "ch:1:sloid:5300",  "stop_name": "Lugano"},
]

local_stations = [
    {"city": "Zürich",     "stop_id": "ch:1:sloid:87348", "stop_name": "Zürich, Bahnhofplatz/HB"},
    {"city": "Bern",       "stop_id": "ch:1:sloid:76646", "stop_name": "Bern, Bahnhof"},
    {"city": "Basel",      "stop_id": "ch:1:sloid:78143", "stop_name": "Basel, Bahnhof SBB"},
    {"city": "Luzern",     "stop_id": "ch:1:sloid:8450",  "stop_name": "Luzern, Bahnhof"},
    {"city": "St. Gallen", "stop_id": "ch:1:sloid:74095", "stop_name": "St. Gallen, Bahnhof"},
    {"city": "Lausanne",   "stop_id": "ch:1:sloid:92050", "stop_name": "Lausanne, gare"},
    {"city": "Genève",     "stop_id": "ch:1:sloid:87057", "stop_name": "Genève, gare Cornavin"},
    {"city": "Lugano",     "stop_id": "ch:1:sloid:5380",  "stop_name": "Lugano, Stazione"},
]

print("Rail-Stops:", len(rail_stations))
print("Local-Stops:", len(local_stations))


Rail-Stops: 8
Local-Stops: 8


## 5. Gemeinsame Stationskonfiguration

In [9]:
rail_df = pd.DataFrame(rail_stations).assign(station_type="rail")
local_df = pd.DataFrame(local_stations).assign(station_type="local")

all_stations = pd.concat(
    [rail_df, local_df],
    ignore_index=True
)

display(
    all_stations[
        [
            "city",
            "station_type",
            "stop_id",
            "stop_name"
        ]
    ]
)


,city,station_type,stop_id,stop_name
0,Zürich,rail,ch:1:sloid:3000,Zürich HB
1,Bern,rail,ch:1:sloid:7000,Bern
2,Basel,rail,ch:1:sloid:10,Basel SBB
3,Luzern,rail,ch:1:sloid:5000,Luzern
4,St. Gallen,rail,ch:1:sloid:6302,St. Gallen
5,Lausanne,rail,ch:1:sloid:1120,Lausanne
6,Genève,rail,ch:1:sloid:1008,Genève
7,Lugano,rail,ch:1:sloid:5300,Lugano
8,Zürich,local,ch:1:sloid:87348,"Zürich, Bahnhofplatz/HB"
9,Bern,local,ch:1:sloid:76646,"Bern, Bahnhof"


In [10]:
all_results = []

for station in all_stations.to_dict("records"):

    print("\n" + "=" * 60)
    print(
        station["city"],
        "-",
        station["station_type"],
        "-",
        station["stop_name"]
    )
    print("=" * 60)

    try:
        df_station = collect_ojp_station(
            stop_id=station["stop_id"],
            stop_name=station["stop_name"],
            token=TOKEN,
            number_of_results=30
        )

        if not df_station.empty:
            df_station["city"] = station["city"]
            df_station["station_type"] = station["station_type"]

            all_results.append(df_station)

    except Exception as error:
        print("FEHLER:")
        print(error)


Zürich - rail - Zürich HB
Zürich HB: HTTP 200
Zürich HB: 30 Verbindungen
Gespeichert: data/interim/zuerich_hb_20260922_120148.csv

Bern - rail - Bern
Bern: HTTP 200
Bern: 30 Verbindungen
Gespeichert: data/interim/bern_20260922_120148.csv

Basel - rail - Basel SBB
Basel SBB: HTTP 200
Basel SBB: 30 Verbindungen
Gespeichert: data/interim/basel_sbb_20260922_120148.csv

Luzern - rail - Luzern
Luzern: HTTP 200
Luzern: 30 Verbindungen
Gespeichert: data/interim/luzern_20260922_120148.csv

St. Gallen - rail - St. Gallen
St. Gallen: HTTP 200
St. Gallen: 30 Verbindungen
Gespeichert: data/interim/st._gallen_20260922_120148.csv

Lausanne - rail - Lausanne
Lausanne: HTTP 200
Lausanne: 30 Verbindungen
Gespeichert: data/interim/lausanne_20260922_120149.csv

Genève - rail - Genève
Genève: HTTP 200
Genève: 29 Verbindungen
Gespeichert: data/interim/geneve_20260922_120149.csv

Lugano - rail - Lugano
Lugano: HTTP 200
Lugano: 30 Verbindungen
Gespeichert: data/interim/lugano_20260922_120149.csv

Zürich - lo